# MediVision AI — Predictive Health Analytics (ML Module v1)

**Objective:** build the first ML component of MediVision AI using the CDC/NCHS NHANES 2017–March 2020 pre-pandemic public-use data.

### Product direction
The first prototype is intentionally **general-health oriented**, not diabetes-specific. We will build a reusable tabular ML pipeline and evaluate several common-condition screening targets:

- Hypertension history
- Diabetes history
- Coronary heart disease history

> **Important clinical framing:** NHANES is cross-sectional. These labels represent reported/current health conditions, so this first model is a **screening/condition-classification prototype**, not a validated prospective disease-onset predictor and not a diagnostic tool.

The project proposal calls for EDA, missing-value handling, feature engineering, model training/validation, and later FastAPI integration. This notebook starts that pipeline.


## Official data sources

All raw data below are from CDC/NCHS NHANES 2017–March 2020 pre-pandemic public-use files.

- NHANES landing page: https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?Cycle=2017-2020
- Demographics: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DEMO.xpt
- Body Measures: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BMX.xpt
- Blood Pressure & Cholesterol questionnaire: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BPQ.xpt
- Blood Pressure examination: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BPXO.xpt
- Physical Activity: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_PAQ.xpt
- Smoking: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_SMQ.xpt
- Current Health Status: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_HSQ.xpt
- Diabetes questionnaire: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DIQ.xpt
- Medical Conditions: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_MCQ.xpt
- Glycohemoglobin (HbA1c): https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_GHB.xpt
- Plasma fasting glucose: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_GLU.xpt
- HDL: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_HDL.xpt
- Total cholesterol: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_TCHOL.xpt
- LDL + triglycerides: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_TRIGLY.xpt


In [5]:
# Core libraries
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, f1_score, precision_score, recall_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DATA_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Ready:", DATA_DIR.resolve())


Ready: D:\depi_graduation_project\MediVision-AI\data\raw


In [6]:
# Download the selected NHANES files.
# Run this cell once in an environment with internet access.
# If you already downloaded the files manually, the cell will skip them.

import requests
from tqdm.auto import tqdm

FILES = {
    "P_DEMO.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DEMO.xpt",
    "P_BMX.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BMX.xpt",
    "P_BPXO.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BPXO.xpt",
    "P_BPQ.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BPQ.xpt",
    "P_PAQ.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_PAQ.xpt",
    "P_SMQ.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_SMQ.xpt",
    "P_HSQ.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_HSQ.xpt",
    "P_DIQ.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DIQ.xpt",
    "P_MCQ.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_MCQ.xpt",
    "P_GHB.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_GHB.xpt",
    "P_GLU.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_GLU.xpt",
    "P_HDL.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_HDL.xpt",
    "P_TCHOL.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_TCHOL.xpt",
    "P_TRIGLY.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_TRIGLY.xpt",
}

for filename, url in FILES.items():
    path = DATA_DIR / filename
    if path.exists() and path.stat().st_size > 0:
        print(f"Exists: {filename}")
        continue

    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    with open(path, "wb") as f:
        with tqdm(total=total, unit="B", unit_scale=True, desc=filename) as pbar:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

print("Download step complete.")


P_DEMO.xpt: 3.61MB [00:00, 5.54MB/s]
P_BMX.xpt: 2.52MB [00:00, 11.0MB/s]
P_BPXO.xpt: 1.04MB [00:00, 6.94MB/s]
P_BPQ.xpt: 900kB [00:00, 17.5MB/s]
P_PAQ.xpt: 1.32MB [00:00, 5.44MB/s]
P_SMQ.xpt: 1.43MB [00:00, 414MB/s]
P_HSQ.xpt: 152kB [00:00, 13.2MB/s]                    
P_DIQ.xpt: 3.36MB [00:00, 27.2MB/s]
P_MCQ.xpt: 7.56MB [00:00, 24.3MB/s]
P_GHB.xpt: 168kB [00:00, 4.18MB/s]                    
P_GLU.xpt: 164kB [00:00, 5.07MB/s]
P_HDL.xpt: 294kB [00:00, 9.02MB/s]
P_TCHOL.xpt: 294kB [00:00, 8.84MB/s]
P_TRIGLY.xpt: 409kB [00:00, 4.42MB/s]

Download step complete.


In [7]:
# Load all downloaded XPT files
def read_xpt(filename):
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"{filename} is missing. Run the download cell first or place it in {DATA_DIR}."
        )
    return pd.read_sas(path, format="xport")

raw = {name.replace(".xpt", ""): read_xpt(name) for name in FILES}

for name, df in raw.items():
    print(f"{name:12s} shape={df.shape}")


P_DEMO       shape=(15560, 29)
P_BMX        shape=(14300, 22)
P_BPXO       shape=(11656, 12)
P_BPQ        shape=(10195, 11)
P_PAQ        shape=(9693, 17)
P_SMQ        shape=(11137, 16)
P_HSQ        shape=(9445, 2)
P_DIQ        shape=(14986, 28)
P_MCQ        shape=(14986, 63)
P_GHB        shape=(10409, 2)
P_GLU        shape=(5090, 4)
P_HDL        shape=(12198, 3)
P_TCHOL      shape=(12198, 3)
P_TRIGLY     shape=(5090, 10)


In [8]:
# Merge the tables on SEQN (NHANES respondent ID)
# We keep the merge logic explicit so every feature source remains traceable.

df = raw["P_DEMO"].copy()

merge_order = [
    "P_BMX", "P_BPXO", "P_BPQ", "P_PAQ", "P_SMQ", "P_HSQ",
    "P_DIQ", "P_MCQ", "P_GHB", "P_GLU", "P_HDL", "P_TCHOL", "P_TRIGLY"
]

for name in merge_order:
    before = len(df)
    df = df.merge(raw[name], on="SEQN", how="left", suffixes=("", f"_{name}"))
    print(f"{name}: {before:,} -> {len(df):,} rows")

print("\nFinal shape:", df.shape)


P_BMX: 15,560 -> 15,560 rows
P_BPXO: 15,560 -> 15,560 rows
P_BPQ: 15,560 -> 15,560 rows
P_PAQ: 15,560 -> 15,560 rows
P_SMQ: 15,560 -> 15,560 rows
P_HSQ: 15,560 -> 15,560 rows
P_DIQ: 15,560 -> 15,560 rows
P_MCQ: 15,560 -> 15,560 rows
P_GHB: 15,560 -> 15,560 rows
P_GLU: 15,560 -> 15,560 rows
P_HDL: 15,560 -> 15,560 rows
P_TCHOL: 15,560 -> 15,560 rows
P_TRIGLY: 15,560 -> 15,560 rows

Final shape: (15560, 209)


# EDA